# 第 3 章 01：马尔科夫决策过程的基本概念

这一课只解决一个问题：**怎样用一个数学模型描述“智能体做一步决策，环境给出下一步结果”？** 你已经在 GridWorld 中见过状态、动作、奖励与回报；现在把它们整理成 MDP。读完应能说清 MDP 的组成、马尔科夫性质，以及策略和环境转移各负责什么。


## 1. 从一次交互说起

假设智能体站在门前。它先观察当前状态 $s_t$，选择动作 $a_t$；环境随后给出奖励 $r_{t+1}$ 和下一状态 $s_{t+1}$。这一步可以记成

$$
s_t\xrightarrow{\text{选择 }a_t}(r_{t+1},s_{t+1}).
$$

只列出一条已经发生的轨迹，还不能预测其他可能的结果。MDP 要补上：**当前可以选哪些动作？每个动作可能把环境带到哪里？各有多大概率？会得到什么奖励？**


## 2. “马尔科夫”是什么意思？

马尔科夫性质说：如果当前状态 $s_t$ 已经概括了做下一步预测所需的信息，那么知道更早的历史，不会再改变“下一状态和奖励”的概率。简写成：

$$
P(s_{t+1},r_{t+1}\mid s_t,a_t,\text{过去的历史})
=P(s_{t+1},r_{t+1}\mid s_t,a_t).
$$

这里的意思**不是**环境完全确定；下一步仍可能随机。它说的是：给定当前状态和动作后，旧历史不再提供额外信息。

例如迷宫有一扇锁门。若“状态”只记录位置，同样站在门前，是否拿过钥匙会改变下一步能否开门；这个状态就不够用。改成 $s=(\text{位置},\text{是否有钥匙})$ 后，当前状态才包含关键历史信息。这是**设计状态**时最重要的检查方法。


## 3. MDP 的五个部件

一个常用的 MDP 记法是 $(\mathcal S,\mathcal A,P,R,\gamma)$：

| 符号 | 问什么？ | 门前例子 |
| --- | --- | --- |
| $\mathcal S$ 状态空间 | 可能处于哪些状态？ | 门前、终点 |
| $\mathcal A$ 动作空间 | 可以做什么？ | 向左、向右 |
| $P$ 状态转移规律 | 做完动作，下一个状态有多大概率是什么？ | 向右后有机会到终点，也可能留在门前 |
| $R$ 奖励规律 | 这一步获得什么反馈？ | 到终点得 $+10$，未到达得 $-1$ |
| $\gamma$ 折扣因子 | 怎样权衡稍后的奖励？ | $\gamma=0.9$ 时下一步奖励乘 $0.9$ |

状态空间、动作空间、转移和奖励描述任务本身；折扣因子规定如何计算长期回报。**策略 $\pi$ 是智能体选择动作的规则，通常是要学习的对象，不是上面五元组中的环境转移规律。**


## 4. 状态和动作是两种不同的东西

状态 $s\in\mathcal S$ 描述智能体当前面对的局面；动作 $a\in\mathcal A$ 是它此刻能作出的选择。在实际任务中，可选动作也可能随状态变化，此时记为 $\mathcal A(s)$。

若终点是回合终止状态，到达终点后这一局不再继续。终止状态帮助我们明确“从哪里开始、什么时候结束”，但它不会改变上面“观察状态再选择动作”的基本交互顺序。


## 5. 转移概率 $P$：动作相同，结果也可能不同

在门前状态 $s$ 选择“向右”，假设有 $0.8$ 的概率到达终点 $g$，有 $0.2$ 的概率被挡住、仍留在 $s$：

$$
P(g\mid s,\text{右})=0.8,\qquad P(s\mid s,\text{右})=0.2.
$$

这里 $P(s'\mid s,a)$ 的意思是：**已知当前状态是 $s$、动作是 $a$，下一状态为 $s'$ 的概率**。对固定的 $(s,a)$，所有可能下一状态的概率要相加为 $1$。在这个例子里就是 $0.8+0.2=1$。

如果“向左”总会留在门前，则 $P(s\mid s,\text{左})=1$。注意：动作是智能体选的；动作之后会发生哪种结果，是环境的转移规律决定的。


## 6. 奖励 $R$：一次实际奖励与平均奖励

为保持例子简单，规定向右成功到终点得 $+10$，失败留在门前得 $-1$。一次真正的尝试，只会观察到其中一个奖励；在动作发生前，可以算它的**期望即时奖励**：

$$
\mathbb E[r_{t+1}\mid s_t=s,a_t=\text{右}]
=0.8\times10+0.2\times(-1)=7.8.
$$

$7.8$ 是重复许多次相同局面和动作时的平均值，不是某一次实际收到的奖励。不同教材可能把 $R$ 写作 $R(s,a,s')$、$R(s,a)$ 或奖励的条件分布；这里先把“到达哪儿得到什么奖励”作为规则，而用上式算平均值。


## 7. 折扣因子 $\gamma$：把多步奖励连起来

MDP 不只关心下一步。令当前这一步开始的折扣回报为

$$
G_t=r_{t+1}+\gamma r_{t+2}+\gamma^2 r_{t+3}+\cdots.
$$

若两步奖励依次是 $-1$ 和 $+10$，取 $\gamma=0.9$，则这条轨迹从第一步起的回报为 $-1+0.9\times10=8$。折扣因子越小，越看重近期奖励。有限回合中可以讨论 $\gamma=1$；无限持续任务通常取 $0\leq\gamma<1$，避免无穷回报的求和问题。

**即时奖励**是一步的反馈；**回报**是沿着未来轨迹累计后的结果。不要把两者混为一谈。


## 8. 策略 $\pi$：智能体自己怎样选动作

在门前状态 $s$，策略可以规定 $\pi(\text{右}\mid s)=0.75$、$\pi(\text{左}\mid s)=0.25$。两项相加为 $1$；这是一个关于**动作选择**的 PMF。

转移概率 $P(g\mid s,\text{右})=0.8$ 则描述**已经选了向右之后**，环境让智能体到达终点的概率。两层随机性要分开。若只有“向右”可能到终点，按上述策略行动时，下一步到终点的总概率是

$$
P(g\mid s,\pi)=\pi(\text{右}\mid s)P(g\mid s,\text{右})
=0.75\times0.8=0.6.
$$

这一步连接了上一章的 PMF、条件概率与期望：先按策略抽动作，再按环境规律抽结果。


## 9. 用一句话复盘这一步

智能体看见 $s_t$，按策略 $\pi$ 选择 $a_t$；环境按 $P$ 给出 $s_{t+1}$，并按奖励规律给出 $r_{t+1}$；之后重复。状态若足够概括过去，这一轮交互就可以用 MDP 的方式描述。

学习强化学习算法时，我们希望找到能使**期望回报**较高的策略，而不是保证每次尝试都得到最好的单次结果。价值函数、贝尔曼方程和具体算法将在后续笔记中逐步引入。


## 10. 自检：先不看答案

1. 为什么只用“角色的位置”表示锁门迷宫的状态可能不满足马尔科夫性质？怎样改状态？
2. 在门前向右时，到终点概率是 $0.8$，留在门前概率是 $0.2$。$P$ 在描述谁的规律？
3. 向右成功得 $+10$、失败得 $-1$。期望即时奖励是多少？这是不是某一次必然收到的奖励？
4. 策略向右概率为 $0.75$，向左概率为 $0.25$。若只有向右能到终点，下一步到终点概率是多少？
5. 即时奖励与折扣回报分别回答什么问题？


### 参考答案

1. 因为是否有钥匙会影响开门结果；可用 $(\text{位置},\text{是否有钥匙})$ 表示状态。
2. $P$ 描述环境在给定状态和动作之后如何产生下一状态。
3. $0.8\times10+0.2\times(-1)=7.8$；某一次实际收到的是 $+10$ 或 $-1$。
4. $0.75\times0.8=0.6$。
5. 即时奖励是一轮交互的反馈；回报把这一轮及之后的奖励按折扣加起来。
